In [2]:

import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
device = torch.device("cpu")
# Load corrected dataset
saved = joblib.load("Results/Leakage_Free/final_corrected_sequences.joblib")
X_corrected = saved["X"]
y_corrected = saved["y"]
indices_corrected = saved["indices"]
window = saved["window"]
train_end = saved["train_end"]
train_mask = indices_corrected < train_end
test_mask = indices_corrected >= train_end
# Restore training-only scaling
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_raw = X_corrected[train_mask]
X_test_raw = X_corrected[test_mask]

y_train_raw = y_corrected[train_mask].reshape(-1, 1)
y_test_raw = y_corrected[test_mask].reshape(-1, 1)

X_train = scaler_X.fit_transform(
    X_train_raw.reshape(-1, 1)
).reshape(-1, window, 1)

X_test = scaler_X.transform(
    X_test_raw.reshape(-1, 1)
).reshape(-1, window, 1)

y_train = scaler_y.fit_transform(
    y_train_raw
).flatten()

y_test = scaler_y.transform(
    y_test_raw
).flatten()

test_indices = indices_corrected[test_mask]

# Recreate the exact saved model architecture
class CorrectedITransformer(nn.Module):
    def __init__(self, seq_len=10, num_features=1):
        super().__init__()

        self.embedding = nn.Linear(seq_len, 32)

        layer = nn.TransformerEncoderLayer(
            d_model=32,
            nhead=4,
            dim_feedforward=64,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=2,
            enable_nested_tensor=False
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_features * 32, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.embedding(x)
        x = self.encoder(x)
        return self.head(x).squeeze(-1)


itransformer_model = CorrectedITransformer(
    seq_len=window,
    num_features=1
).to(device)

itransformer_model.load_state_dict(
    torch.load(
        "Results/Leakage_Free/iTransformer/model.pth",
        map_location=device,
        weights_only=True
    )
)

itransformer_model.eval()

# Restore corrected completion timestamps
data = joblib.load("Dataset/cycle_time_data.joblib")
df_fixed = data["df"].copy()

df_fixed["Scheduled_Start"] = pd.to_datetime(
    df_fixed["Scheduled_Start"], errors="coerce"
)

df_fixed["Actual_Start"] = pd.to_datetime(
    df_fixed["Actual_Start"], errors="coerce"
)

target = np.asarray(data["yraw"], dtype=float).ravel()

df_fixed["Corrected_End"] = (
    df_fixed["Actual_Start"]
    + pd.to_timedelta(target, unit="m")
)

print("Corrected iTransformer loaded successfully.")
print("Training sequences:", len(X_train))
print("Testing sequences:", len(X_test))
print("Model parameters:",
      sum(p.numel() for p in itransformer_model.parameters()))


Corrected iTransformer loaded successfully.
Training sequences: 382
Testing sequences: 595
Model parameters: 19617


In [3]:

# STEP 23: Delayed-feedback iTransformer + ADWIN

import os, time, copy, heapq, random
import numpy as np
import pandas as pd
import torch
from river.drift import ADWIN
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

outdir = "Results/Leakage_Free/iTransformer_ADWIN_Delayed"
os.makedirs(outdir, exist_ok=True)

# Load original trained baseline weights
adaptive = copy.deepcopy(itransformer_model).to(device)
adaptive.load_state_dict(
    torch.load(
        "Results/Leakage_Free/iTransformer/model.pth",
        map_location=device,
        weights_only=True
    )
)
adaptive.eval()

detector = ADWIN(delta=0.002)

# Chronological test data
order = np.argsort(test_indices)
idx = test_indices[order]
Xt = X_test[order]
yt = y_test[order]

# Forecast and completion timestamps
starts = pd.to_datetime(
    df_fixed.iloc[idx]["Scheduled_Start"]
).to_numpy()

ends = pd.to_datetime(
    df_fixed.iloc[idx]["Corrected_End"]
).to_numpy()

assert pd.Series(starts).is_monotonic_increasing
assert pd.notna(starts).all()

# Pending feedback: (completion time, sequence number)
pending = []
completed = []

predictions = np.zeros(len(idx))
actuals = y_test_raw[order, 0].copy()
detections = []
adaptation_points = []

def adapt_completed(model, completed_positions):
    recent = completed_positions[-100:]

    xb = torch.tensor(
        Xt[recent], dtype=torch.float32
    )
    yb = torch.tensor(
        yt[recent], dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(xb, yb),
        batch_size=16,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.0001
    )
    criterion = torch.nn.MSELoss()

    model.train()
    for _ in range(10):
        for xbatch, ybatch in loader:
            optimizer.zero_grad()
            loss = criterion(
                model(xbatch.to(device)),
                ybatch.to(device)
            )
            loss.backward()
            optimizer.step()
    model.eval()

def process_feedback(pos):
    completed.append(pos)

    error = abs(
        actuals[pos] - predictions[pos]
    )

    detector.update(float(error))

    if detector.drift_detected:
        detections.append(int(idx[pos]))
        adapt_completed(adaptive, completed)
        adaptation_points.append(int(idx[pos]))

start_time = time.time()

for pos in range(len(idx)):
    forecast_time = starts[pos]

    # Process jobs completed before this forecast
    while pending and pending[0][0] <= forecast_time:
        _, old_pos = heapq.heappop(pending)
        process_feedback(old_pos)

    # Forecast with currently available model
    adaptive.eval()
    with torch.no_grad():
        p_scaled = adaptive(
            torch.tensor(
                Xt[pos:pos+1],
                dtype=torch.float32,
                device=device
            )
        ).cpu().numpy().reshape(-1, 1)

    predictions[pos] = scaler_y.inverse_transform(
        p_scaled
    )[0, 0]

    # Actual target is released only at completion
    if pd.notna(ends[pos]):
        assert ends[pos] >= starts[pos]
        heapq.heappush(
            pending, (ends[pos], pos)
        )

elapsed = time.time() - start_time

# Do not process remaining feedback:
# no later test forecasts can benefit from it.

rows = []
for name, (a, b) in {
    "Stable": (400, 600),
    "Drift": (600, 800),
    "Recovery": (800, 995),
    "Overall": (400, 995)
}.items():
    mask = (idx >= a) & (idx < b)
    error = actuals[mask] - predictions[mask]

    rows.append({
        "Period": name,
        "Samples": int(mask.sum()),
        "MAE": round(np.mean(np.abs(error)), 3),
        "RMSE": round(np.sqrt(np.mean(error**2)), 3)
    })

metrics = pd.DataFrame(rows)

pd.DataFrame({
    "Index": idx,
    "Actual": actuals,
    "Predicted": predictions
}).to_csv(
    f"{outdir}/predictions.csv", index=False
)

metrics.to_csv(
    f"{outdir}/metrics.csv", index=False
)

torch.save(
    adaptive.state_dict(),
    f"{outdir}/model.pth"
)

print("FINAL DELAYED-FEEDBACK RESULTS")
display(metrics)

print("Detected observations:", detections)
print("Total adaptations:", len(adaptation_points))
print("Execution time:", round(elapsed, 2), "seconds")
print("Results saved successfully.")


FINAL DELAYED-FEEDBACK RESULTS


,Period,Samples,MAE,RMSE
0,Stable,200,27.725,33.701
1,Drift,200,55.636,67.036
2,Recovery,195,27.931,34.671
3,Overall,595,37.175,47.815


Detected observations: [662, 872]
Total adaptations: 2
Execution time: 5.03 seconds
Results saved successfully.
